Setup

In [ ]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az 
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("domestic_well_radon.csv")
df = df.dropna(subset=["log_radon", "well_depth_ft", "lithology", "aquifer"])
df["log_depth"] = np.log(df["well_depth_ft"])
df["centered_log_depth"] = (df["log_depth"] - df["log_depth"].mean())
df = df[["log_radon", "centered_log_depth", "lithology", "aquifer"]].reset_index(drop=True)
print("Number of lithologies:", df["lithology"].nunique())
print("Number of aquifers:", df["aquifer"].nunique())
print("Wells with complete data:", len(df))
print(df.groupby("lithology")["log_radon"].agg(["count", "mean", "std", "min", "max"]))
df.head()

Priors

In [ ]:
intercept = {
    "Intercept": bmb.Prior("Normal", mu=6.0, sigma=3.0)
}

depth = {
    "centered_log_depth": bmb.Prior("Normal", mu=0.0, sigma=1.0),
}

sigma = {
    "sigma": bmb.Prior("HalfNormal", sigma=1.0)
}

lithology_intercept = {
    "1|lithology": bmb.Prior("Normal", mu=0.0, sigma=bmb.Prior("HalfNormal", sigma=1.0))
}

aquifier_intercept = {
    "1|aquifer": bmb.Prior("Normal", mu=0.0, sigma=bmb.Prior("HalfNormal", sigma=1.0)),
}

varying_sigma_prior = {
    "sigma": {
        "lithology": bmb.Prior("Normal", mu=0.0, sigma=1.0),
    }
}
lithology_specific_depth =  {
    "centered_log_depth|lithology": bmb.Prior(
        "Normal", mu=0.0, sigma=bmb.Prior("HalfNormal", sigma=0.5)
    )
}

nu = {
    "nu": bmb.Prior("Gamma", alpha=2, beta=0.1)
}

pooled_priors = intercept | depth | sigma

aquifer_hierarchical_priors = intercept | depth | sigma | aquifier_intercept

combined_hierarchical_priors = intercept | depth | sigma | lithology_intercept | aquifier_intercept

combined_varying_sigma_hierarchical_priors = intercept | depth | lithology_intercept | aquifier_intercept | varying_sigma_prior
 
lithology_specific_depth_coefficient_priors = intercept | depth | sigma | lithology_specific_depth | aquifier_intercept

combined_varying_sigma_hierarchical_priors_t = intercept | depth | lithology_intercept | aquifier_intercept | varying_sigma_prior | nu

pooled_priors_t = intercept | depth | sigma | nu

Models

In [ ]:
combined_hierarchical_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | lithology) + (1 | aquifer)",
    ),
    data=df,
    priors=combined_hierarchical_priors,
)

combined_varying_sigma_hierarchical_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | lithology) + (1 | aquifer)",
        "sigma ~ 0 + lithology",
    ),
    data=df,
    priors=combined_varying_sigma_hierarchical_priors,
)


combined_varying_sigma_hierarchical_model_t = bmb.Model(
    bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | lithology) + (1 | aquifer)",
        "sigma ~ 0 + lithology",
    ),
    data=df,
    family="t",
    priors=combined_varying_sigma_hierarchical_priors_t,
)
aquifer_hierarchical_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | aquifer)",
    ),
    data=df,
    priors=aquifer_hierarchical_priors,
)

lithology_specific_depth_coefficient_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (centered_log_depth | lithology) + (1 | aquifer)",
    ),
    data=df,
    priors=lithology_specific_depth_coefficient_priors,
)

pooled_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth",
    ),
    data=df,
    priors=pooled_priors,
)

pooled_model_t = bmb.Model(
    bmb.Formula("log_radon ~ 1 + centered_log_depth"),
    data=df,
    family="t",
    priors=pooled_priors_t,
)

Fit all models.

In [ ]:
def fit_model(model, **sampler_kwargs):
    fit = model.fit(**sampler_kwargs)
    model.compute_log_likelihood(fit)      # log_likelihood for LOO-PIT
    model.predict(fit, kind="response")    # for posterior predictive checks 
    return fit


model_specs = {
    # name: (model, sampler kwargs)
    "pooled": (pooled_model, dict(target_accept=0.8)),
    #"pooled_t": (pooled_model_t, dict(target_accept=0.8)),
    "aquifer_hierarchical": (aquifer_hierarchical_model, dict(draws=4000, target_accept=0.8)),
    "combined_hierarchical": (combined_hierarchical_model, dict(target_accept=0.95)),
    "lithology_specific_depth_coefficient": (lithology_specific_depth_coefficient_model, dict(target_accept=0.95)),
    "combined_varying_sigma_hierarchical": (combined_varying_sigma_hierarchical_model, dict(target_accept=0.95)),
    "combined_varying_sigma_hierarchical_t": (combined_varying_sigma_hierarchical_model_t, dict(target_accept=0.95)),
}

fits = {}
for name, (model, kwargs) in model_specs.items():
    print(f"Fitting model: {name}")
    fits[name] = fit_model(model, **kwargs)

Convergence statistics:

In [ ]:
def analyze_fits(fits):
    """Detailed fit diagnostics"""
    for name, fit in fits.items():
        params = [v for v in fit.posterior.data_vars if "__obs__" not in fit.posterior[v].dims]  # Hides model internal params
        print(f"Diagnostics for: {name}")
        print("Divergences:", int(fit.sample_stats["diverging"].sum()))
        print(az.summary(fit, var_names=params, round_to="none"))
        loo = az.loo(fit, pointwise=True)
        print(f"LOO: {name}")
        print(loo)
        print()

def convergence_summary(fits):
    """Divergences, worst R-hat and smallest ESS over all parameters, per model."""
    rows = []
    for name, fit in fits.items():
        params = [v for v in fit.posterior.data_vars if "__obs__" not in fit.posterior[v].dims] # Hides model internal params
        summary = az.summary(fit, var_names=params, kind="diagnostics", round_to="none")
        rows.append({
            "model": name,
            "divergences": int(fit.sample_stats["diverging"].sum()),
            "max_rhat": summary["r_hat"].max(),
            "max_rhat_param": summary["r_hat"].idxmax(),
            "min_ess_bulk": summary["ess_bulk"].min(),
            "min_ess_bulk_param": summary["ess_bulk"].idxmin(),
            "min_ess_tail": summary["ess_tail"].min(),
            "min_ess_tail_param": summary["ess_tail"].idxmin(),
        })
    out = pd.DataFrame(rows).set_index("model")
    out["max_rhat"] = out["max_rhat"].round(3)
    out["min_ess_bulk"] = out["min_ess_bulk"].round(0).astype(int)
    out["min_ess_tail"] = out["min_ess_tail"].round(0).astype(int)
    return out

with pd.option_context("display.width", 200, "display.max_columns", None):
    print(convergence_summary(fits))
    
#analyze_fits(fits)

In [ ]:

def compare_models(fits):
    comparison = az.compare(fits)

    with pd.option_context("display.width", 200, "display.max_columns", None):
        print(comparison[["rank", "elpd", "p", "elpd_diff", "dse", "p_worse"]])
    az.plot_compare(comparison)
    return comparison


comparison = compare_models(fits)

Add some predictive checks.

In [ ]:

def plot_predictive_checks(fit, obs_idx=None, title=None):
    """Posterior predictive KDE and LOO-PIT D-ECDF."""
    coords = {"__obs__": obs_idx} if obs_idx is not None else None  #  For indexing observations, used for different lithologies later. 
    pc = az.combine_plots(
        fit,
        plots=[
            (az.plot_ppc_dist, {"kind": "kde", "num_samples": 20}),
            (az.plot_loo_pit, {}),
        ],
        group="posterior_predictive",
        var_names="log_radon",
        coords=coords,
        plot_names=["Posterior predictive KDE", "LOO-PIT D-ECDF"],
        figure_kwargs={"figsize": (12, 3.6)},
    )
    fig = pc.viz["figure"].item()
    if title is not None:
        fig.suptitle(title, fontsize=14)
    fig.set_layout_engine("constrained")
    plt.show()

for name, (model, kwargs) in model_specs.items():
     plot_predictive_checks(fits[name], title=name)


Add lithograpgy specific checks for good models

In [ ]:
good_models = ["combined_varying_sigma_hierarchical", "combined_varying_sigma_hierarchical_t"]

for lith in df["lithology"].unique():
    idx = np.where(df["lithology"].values == lith)[0]
    for name in good_models:
        plot_predictive_checks(fits[name], obs_idx=idx, title=f"{lith} (n = {len(idx)}) — {name}")
        